In [1]:
# ==============================================================================
# Machine Learning Telco Churn Predictor (Synchronized Pipeline)
# ==============================================================================

# ------------------------------------------------------------------------------
# 1. Dependency Imports & Setup
# ------------------------------------------------------------------------------
import os
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.calibration import CalibratedClassifierCV

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)

# ------------------------------------------------------------------------------
# 2. Dynamic Cross-Platform Path Setup
# ------------------------------------------------------------------------------
BASE_DIR = Path.cwd()
MODEL_DIR = BASE_DIR / "model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------------------
# 3. Data Ingestion, Data Cleaning & Feature Engineering
# ------------------------------------------------------------------------------
data_path = BASE_DIR / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
if not data_path.exists():
    raise FileNotFoundError(f"Missing required dataset: '{data_path.name}' in {BASE_DIR}")

df = pd.read_csv(data_path)

if "customerID" in df.columns:
    df.drop(columns=["customerID"], inplace=True)

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].astype(str).str.strip(), errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Domain-specific ratio features
df['ChargeRatio'] = df['MonthlyCharges'] / (df['TotalCharges'] + 1)
df['AvgMonthlySpend'] = df['TotalCharges'] / (df['tenure'] + 1)

# ------------------------------------------------------------------------------
# 4. Categorical Feature Encoding & Serialization
# ------------------------------------------------------------------------------
categorical_cols = df.select_dtypes(include=['object']).columns

# Apply One-Hot Encoding to nominal variables (>2 unique categories)
df = pd.get_dummies(df, columns=[c for c in categorical_cols if df[c].nunique() > 2], drop_first=True)

# Convert boolean columns to integer (0/1) to prevent CSV formatting issues
bool_cols = df.select_dtypes(include=['bool']).columns
df[bool_cols] = df[bool_cols].astype(int)

# Apply LabelEncoder to binary text variables
binary_cols = df.select_dtypes(include=['object']).columns
encoders = {}
for col in binary_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

joblib.dump(encoders, MODEL_DIR / "encoders.pkl")

# ------------------------------------------------------------------------------
# 5. Stratified Train-Test Dataset Splitting & Exporting test_data.csv
# ------------------------------------------------------------------------------
X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Create test dataset retaining feature structures and ground truth target column
test_data = X_test.copy()
test_data['Churn'] = y_test.values

# Save as standard clean CSV with utf-8-sig encoding for seamless parsing
test_data.to_csv(
    BASE_DIR / "test_data.csv", 
    index=False, 
    encoding='utf-8-sig', 
    lineterminator='\n'
)

# ------------------------------------------------------------------------------
# 6. Feature Scaling & Scaler Serialization
# ------------------------------------------------------------------------------
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

joblib.dump(scaler, MODEL_DIR / "scaler.pkl")
print("✅ Generated clean test_data.csv (with 'Churn' target), model/encoders.pkl, and model/scaler.pkl!\n")

# ------------------------------------------------------------------------------
# 7. Model Instantiation, Training & Evaluation Loop (Fixed Threshold = 0.45)
# ------------------------------------------------------------------------------
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    "Decision Tree Classifier": DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42),
    "K-Nearest Neighbor Classifier": KNeighborsClassifier(n_neighbors=7),
    "Naive Bayes Classifier - Gaussian": GaussianNB(),
    "Support Vector Machine": CalibratedClassifierCV(
        estimator=SVC(class_weight='balanced', random_state=42), 
        ensemble=False
    )
}

filename_map = {
    "Logistic Regression": "logistic_regression.pkl",
    "Decision Tree Classifier": "decision_tree.pkl",
    "K-Nearest Neighbor Classifier": "knn.pkl",
    "Naive Bayes Classifier - Gaussian": "naive_bayes.pkl",
    "Support Vector Machine": "support_vector_machine.pkl"
}

results = []
OPTIMIZED_THRESHOLD = 0.45

for name, model in models.items():
    if name in ["Logistic Regression", "K-Nearest Neighbor Classifier", "Support Vector Machine"]:
        model.fit(X_train_scaled, y_train)
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_proba = model.predict_proba(X_test)[:, 1]
    
    # Classification at fixed 0.45 threshold
    y_pred = (y_proba >= OPTIMIZED_THRESHOLD).astype(int)
    
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    mcc = matthews_corrcoef(y_test, y_pred)
    
    results.append({
        "ML Model Name": name,
        "Accuracy": round(acc, 4),
        "AUC Score": round(auc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1 Score": round(f1, 4),
        "Matthews Correlation Coefficient (MCC Score)": round(mcc, 4)
    })
    
    model_filename = MODEL_DIR / filename_map[name]
    joblib.dump(model, model_filename)
    print(f"✅ Saved model artifact: {model_filename.name}")

# ------------------------------------------------------------------------------
# 8. Output Metrics Display
# ------------------------------------------------------------------------------
results_df = pd.DataFrame(results)
results_df.insert(0, "S.No.", range(1, len(results_df) + 1))

def is_notebook():
    try:
        shell = get_ipython().__class__.__name__
        return True
    except NameError:
        return False

if is_notebook():
    from IPython.display import HTML, display
    
    metric_cols = [
        "Accuracy", 
        "AUC Score", 
        "Precision", 
        "Recall", 
        "F1 Score", 
        "Matthews Correlation Coefficient (MCC Score)"
    ]
    
    styled_table = (
        results_df.style
        .hide(axis="index")
        .format({col: "{:.4f}" for col in metric_cols})
        .set_properties(subset=['ML Model Name'], **{'text-align': 'left !important'})
        .set_table_styles([
            {
                'selector': 'table',
                'props': [
                    ('border-collapse', 'collapse'),
                    ('margin', '0')
                ]
            },
            {
                'selector': 'th',
                'props': [
                    ('border', '1px solid #444'),
                    ('padding', '8px 12px'),
                    ('background-color', '#f2f2f2'),
                    ('text-align', 'center'),
                    ('font-weight', 'bold')
                ]
            },
            {
                'selector': 'td',
                'props': [
                    ('border', '1px solid #ccc'),
                    ('padding', '8px 12px'),
                    ('text-align', 'center')
                ]
            }
        ])
    )
    
    display(HTML('<div style="height: 20px;"></div>'))
    display(styled_table)
else:
    print("\n" + results_df.to_string(index=False))

✅ Generated clean test_data.csv (with 'Churn' target), model/encoders.pkl, and model/scaler.pkl!

✅ Saved model artifact: logistic_regression.pkl
✅ Saved model artifact: decision_tree.pkl
✅ Saved model artifact: knn.pkl
✅ Saved model artifact: naive_bayes.pkl
✅ Saved model artifact: support_vector_machine.pkl


S.No.,ML Model Name,Accuracy,AUC Score,Precision,Recall,F1 Score,Matthews Correlation Coefficient (MCC Score)
1,Logistic Regression,0.7239,0.8486,0.4882,0.8316,0.6152,0.4583
2,Decision Tree Classifier,0.6941,0.8271,0.4583,0.8369,0.5922,0.4236
3,K-Nearest Neighbor Classifier,0.7672,0.7909,0.5665,0.5241,0.5444,0.3890
4,Naive Bayes Classifier - Gaussian,0.6537,0.8202,0.4254,0.8690,0.5712,0.3943
5,Support Vector Machine,0.7715,0.8263,0.5580,0.6684,0.6083,0.4524
